# Benchmark Training & Evaluation

Train 4 mô hình benchmark trên dataset forecasting, đánh giá và so sánh.

## 1. Install & Import

In [ ]:
!pip install -q polars xgboost lightgbm scikit-learn joblib matplotlib huggingface_hub

In [ ]:
import os
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
import joblib

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
import xgboost as xgb
import lightgbm as lgb

print('All imports OK')

## 2. Config

In [ ]:
# --- Dataset config ---
HF_REPO_ID = "letung373/building-forecasting-data"  # Hugging Face dataset repo
DATASET_SUBFOLDER = "h24_energy"  # Options: h24_energy, h24_historical_weather, h24_forecast_weather, h168_*, ...

# --- Output dirs (on Drive if Colab, local otherwise) ---
OUTPUT_BASE = "/content/output"  # Change to Drive path if needed
MODELS_DIR = os.path.join(OUTPUT_BASE, "models")
REPORT_DIR = os.path.join(OUTPUT_BASE, "report")

os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)
print(f'Output: {OUTPUT_BASE}')
print(f'Dataset: {HF_REPO_ID} / {DATASET_SUBFOLDER}')

## 3. Load Dataset from Hugging Face

In [ ]:
from huggingface_hub import snapshot_download

local_data = snapshot_download(
    repo_id=HF_REPO_ID,
    repo_type="dataset",
)
DATA_DIR = os.path.join(local_data, "dataset", DATASET_SUBFOLDER)
print(f'Downloaded to: {local_data}')
print(f'Data folder: {DATA_DIR}')

In [ ]:
train = pl.read_parquet(os.path.join(DATA_DIR, "train.parquet"))
val = pl.read_parquet(os.path.join(DATA_DIR, "validation.parquet"))
test = pl.read_parquet(os.path.join(DATA_DIR, "test.parquet"))

print(f'Train:      {train.shape[0]:>10,} rows | {train["timestamp"].min()} → {train["timestamp"].max()}')
print(f'Validation: {val.shape[0]:>10,} rows | {val["timestamp"].min()} → {val["timestamp"].max()}')
print(f'Test:       {test.shape[0]:>10,} rows | {test["timestamp"].min()} → {test["timestamp"].max()}')
print(f'\nFeatures: {train.columns}')

## 4. Prepare Features / Target

In [ ]:
# Columns to exclude from features
EXCLUDE_COLS = {"target", "timestamp", "building_id"}
FEATURE_COLS = [c for c in train.columns if c not in EXCLUDE_COLS]
CATEGORICAL_COLS = [c for c in ["primaryspaceusage", "timezone"] if c in FEATURE_COLS]

print(f'Features ({len(FEATURE_COLS)}): {FEATURE_COLS}')
print(f'Categorical: {CATEGORICAL_COLS}')

In [ ]:
# Convert to pandas for sklearn compatibility
train_pd = train.to_pandas()
val_pd = val.to_pandas()
test_pd = test.to_pandas()

X_train = train_pd[FEATURE_COLS].copy()
y_train = train_pd["target"]
X_val = val_pd[FEATURE_COLS].copy()
y_val = val_pd["target"]
X_test = test_pd[FEATURE_COLS].copy()
y_test = test_pd["target"]

# Encode categorical columns
if CATEGORICAL_COLS:
    encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    X_train[CATEGORICAL_COLS] = encoder.fit_transform(X_train[CATEGORICAL_COLS])
    X_val[CATEGORICAL_COLS] = encoder.transform(X_val[CATEGORICAL_COLS])
    X_test[CATEGORICAL_COLS] = encoder.transform(X_test[CATEGORICAL_COLS])

print(f'X_train: {X_train.shape}')
print(f'X_val:   {X_val.shape}')
print(f'X_test:  {X_test.shape}')

## 5. Train Models

In [ ]:
models = {
    "Linear Regression": LinearRegression(n_jobs=-1),
    "Random Forest": RandomForestRegressor(
        n_estimators=100, n_jobs=-1, random_state=42, max_depth=20
    ),
    "XGBoost": xgb.XGBRegressor(
        n_estimators=500, early_stopping_rounds=20,
        learning_rate=0.05, max_depth=6, random_state=42, n_jobs=-1,
    ),
    "LightGBM": lgb.LGBMRegressor(
        n_estimators=500, early_stopping_rounds=20,
        learning_rate=0.05, max_depth=-1, num_leaves=31,
        random_state=42, n_jobs=-1, verbose=-1,
    ),
}

trained = {}

for name, model in models.items():
    print(f'\nTraining {name} ...')
    if name in ("XGBoost", "LightGBM"):
        model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            verbose=False,
        )
    else:
        model.fit(X_train, y_train)
    trained[name] = model
    print(f'  Done: {name}')

## 6. Evaluate

In [ ]:
def calculate_metrics(y_true, y_pred):
    """Calculate MAE, RMSE, MAPE, SMAPE."""
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100  # percentage
    smape = (
        2 * np.mean(np.abs(y_true - y_pred) / (np.abs(y_true) + np.abs(y_pred))) * 100
    )
    return {"MAE": mae, "RMSE": rmse, "MAPE": mape, "SMAPE": smape}


rows = []
for name, model in trained.items():
    y_val_pred = model.predict(X_val)
    y_test_pred = model.predict(X_test)

    val_metrics = calculate_metrics(y_val.values, y_val_pred)
    test_metrics = calculate_metrics(y_test.values, y_test_pred)

    row = {"Model": name}
    for k, v in val_metrics.items():
        row[f"Val_{k}"] = v
    for k, v in test_metrics.items():
        row[f"Test_{k}"] = v
    rows.append(row)

leaderboard = pl.DataFrame(rows)
leaderboard

In [ ]:
# Save leaderboard
leaderboard.write_csv(os.path.join(REPORT_DIR, "model_leaderboard.csv"))
print(f'Saved: {os.path.join(REPORT_DIR, "model_leaderboard.csv")}')

## 7. Actual vs Predicted Plot

In [ ]:
# Pick best model by Test RMSE
best_name = leaderboard.sort("Test_RMSE")["Model"][0]
best_model = trained[best_name]
y_pred = best_model.predict(X_test)

fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(y_test.values, y_pred, alpha=0.1, s=5)
lims = [min(y_test.values.min(), y_pred.min()), max(y_test.values.max(), y_pred.max())]
ax.plot(lims, lims, "r--", linewidth=1.5, label="y = x")
ax.set_xlabel("Actual")
ax.set_ylabel("Predicted")
ax.set_title(f"Actual vs Predicted — {best_name} (Test)")
ax.legend()
plt.tight_layout()
plt.show()

## 8. Feature Importance

In [ ]:
# Use best tree model for feature importance
tree_model = best_model
if hasattr(tree_model, "feature_importances_"):
    importances = tree_model.feature_importances_
elif hasattr(tree_model, "get_booster"):
    importances = tree_model.get_booster().get_score(importance_type="gain")
    # Map back to full feature list
    importances = np.array([importances.get(f"f{i}", 0) for i in range(len(FEATURE_COLS))])
else:
    importances = None

if importances is not None:
    top_k = 20
    idx = np.argsort(importances)[-top_k:]
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.barh([FEATURE_COLS[i] for i in idx], importances[idx])
    ax.set_xlabel("Importance")
    ax.set_title(f"Top {top_k} Features — {best_name}")
    plt.tight_layout()
    plt.show()
else:
    print(f'{best_name} does not expose feature importances.')

## 9. Save Artifacts

In [ ]:
# Save all models
for name, model in trained.items():
    safe_name = name.lower().replace(" ", "_")
    path = os.path.join(MODELS_DIR, f"{safe_name}.pkl")
    joblib.dump(model, path)
    print(f'Saved: {path}')

# Save best model
best_path = os.path.join(MODELS_DIR, "best_model.pkl")
joblib.dump(best_model, best_path)
print(f'\nBest model ({best_name}): {best_path}')

In [ ]:
# Save evaluation summary
summary_lines = [
    "# Evaluation Summary",
    f"Dataset: {DATASET_SUBFOLDER}",
    f"",
    f"## Leaderboard",
]

# Add leaderboard as markdown table
lb_pd = leaderboard.to_pandas()
summary_lines.append(lb_pd.to_markdown(index=False))

summary_lines.extend([
    f"",
    f"## Best Model",
    f"{best_name}",
])

summary_path = os.path.join(REPORT_DIR, "evaluation_summary.md")
with open(summary_path, "w") as f:
    f.write("\n".join(summary_lines))
print(f'Saved: {summary_path}')